# Experimentul 8: fuziunea rotațiilor cuantice (quantum rotation fusion)

**Pornim exclusiv de la Exp. 3** (ZZ ×1, ansatz ×3, 4 qubiți, 32 de parametri, readout ⟨Z₀⟩) și de la **ponderile lui, neschimbate**. Nu reantrenăm, nu schimbăm feature map-ul, ansatz-ul, topologia CNOT, readout-ul sau encoderul. Singura modificare: porțile de un qubit **consecutive pe același fir** sunt înlocuite cu o singură rotație generală `qml.Rot`, cu aceeași funcție ideală.

## Ipoteze
- **H1:** fuziunea reduce porțile native și locurile de zgomot fără să schimbe funcția fără zgomot (verificat numeric, toleranță 1e-10).
- **H2:** cu mai puține canale depolarizante, scorurile la p = 0.05 sunt mai puțin perturbate (MAE și ρ față de scorurile fără zgomot).
- **H0:** câștigul există doar dacă `Rot` este socotită o singură poartă fizică (Scenariul A). Dacă `Rot` se descompune în rotații elementare (Scenariul B), avantajul dispare sau se inversează.

## De ce ar trebui să ajute (argument matematic, de verificat experimental)
Canalul depolarizant pe un qubit este **covariant unitar**: $D_p(U\rho U^\dagger) = U D_p(\rho) U^\dagger$. Deci, în modelul oficial, perechea „poartă $U_1$, canal, poartă $U_2$, canal” este egală cu „$U_2 U_1$, apoi $D_p \circ D_p$”, iar $D_p \circ D_p = D_q$ cu $1 - \tfrac{4q}{3} = (1 - \tfrac{4p}{3})^2$, adică $q = 2p - \tfrac{4p^2}{3}$. Fuziunea înlocuiește $D_q$ cu $D_p$: pe acele poziții, zgomotul se înjumătățește aproximativ. CNOT-urile și canalele lor rămân identice.

## Protocol
Validation (test **neatins**), 5 seeds, p = 0, 0.001, 0.01, 0.05, `default.mixed`. Metricile dependente de prag folosesc **pragurile salvate de Exp. 3** pentru fiecare (seed, p); pragurile reoptimizate pentru circuitul nou sunt raportate separat, ca analiză secundară (optimistă: alese și evaluate pe același validation).

## 1. Configurație și imports

**Seeds.** Nu există antrenare; `set_seed` fixează totuși numpy și torch, unghiurile aleatoare de verificare folosesc `np.random.default_rng(seed)`, simulatoarele rulează cu `shots=None` (exact, determinist), iar dispozitivul cu shots folosit doar pentru `qml.Tracker` are `seed` fix.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import sklearn
import torch
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ---- arhitectura Exp. 3, neschimbată ----
SPLIT_SEED = 42
N_QUBITS = 4
FEATURE_MAP_REPS = 1
ANSATZ_REPS = 3               # (3 + 1) * 4 * 2 = 32 parametri
SEEDS = [0, 1, 2, 3, 4]
NOISE_LEVELS = [0.001, 0.01, 0.05]
P_ALL = [0.0, *NOISE_LEVELS]
SHOTS_PER_EXECUTION = 1024    # doar pentru cost ledger
BATCH_SIZE = 32               # doar pentru măsurarea qml.Tracker (mini-batch, ca în Exp. 3)

# ---- cele două scenarii: compilare, apoi zgomot după fiecare poartă ----
NATIVE_GATES = {"RX", "RY", "RZ", "Rot", "Hadamard", "CNOT", "CZ"}   # Scenariul A: setul oficial, Rot nativ
ELEMENTARY_GATES = {"RX", "RY", "RZ", "CNOT"}                        # Scenariul B: Rot și H descompuse în rotații elementare
GATE_SETS = {"A": NATIVE_GATES, "B": ELEMENTARY_GATES}
SCENARIO_LABELS = {"A": "A · Rot nativ (oficial)", "B": "B · Rot → RZ·RY·RZ"}

# ---- verificări ----
EQUIV_TOL = 1e-10             # toleranța echivalenței fără zgomot (float64)
REPRO_TOL = 1e-6              # reproducerea scorurilor istorice Exp. 3 / Exp. 6, calculate cu dtype implicit float32 (secțiunea 6b)
TORCH_DEFAULT_DTYPE = torch.float64   # default.mixed creează constante cu dtype-ul implicit torch: float32 dă erori ~1e-7 (secțiunea 6b)
N_VAL_UNITARY = 100           # tranzacții de validation per seed pentru unitare (per bloc și 16×16): 50 fraude + 50 legitime
N_RANDOM = 5000               # unghiuri aleatoare uniforme în [0, π] per seed, pentru expval
SMOKE_ROWS = 2000             # testul de fum: primele tranzacții din validation, seed 0, p = 0.05

EXPERIMENT_NAME = "rotation_fusion"
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir() and (p / "src").is_dir())
RESULTS_DIR = REPO_ROOT / "results" / "result_exp8"
PLOTS_DIR = RESULTS_DIR / "plots"
EXP3_DIR = REPO_ROOT / "results" / "result_exp3"                          # doar citit: ponderi, praguri, metrici
EXP6_SCORES_DIR = REPO_ROOT / "results" / "result_exp6" / "val_scores"    # doar citit: scoruri validation Exp. 3 (p = 0, 0.01, 0.05)
SPLIT_PATH = REPO_ROOT / "split_indices.npz"

ORIG_COLOR, FUSED_COLOR, SEED_DOT_COLOR, CNOT_COLOR = "#2a78d6", "#eb6834", "#8a8f98", "#c9ccd1"
ORIG_LABEL, FUSED_LABEL = "Exp. 3 original", "Exp. 8 fusion"
CIRCUIT_LABELS = {"exp3": ORIG_LABEL, "exp8": FUSED_LABEL}
CIRCUIT_COLORS = {"exp3": ORIG_COLOR, "exp8": FUSED_COLOR}
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SPLIT_SEED)
torch.set_default_dtype(TORCH_DEFAULT_DTYPE)
print(f"seeds: {SEEDS}   p: {P_ALL}   scenarii: {list(GATE_SETS)}   dtype implicit torch: {torch.get_default_dtype()}")
print("pennylane", qml.__version__, "| torch", torch.__version__, "| sklearn", sklearn.__version__)
print("rezultate:", RESULTS_DIR.relative_to(REPO_ROOT))

## 2. Datele și splitul comun

Identic cu Exp. 3: `split_indices.npz`, verificat față de reconstrucția oficială. Din test păstrăm doar indicii pentru verificarea splitului; nici features, nici etichete.

In [ ]:
DATA_PATH = REPO_ROOT / "data" / "creditcard.csv"
for path in [DATA_PATH, SPLIT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
df = pd.read_csv(DATA_PATH)
df["Class"] = df["Class"].astype(int)
assert df.shape == (284807, 31) and "Time" in df.columns and df.index.equals(pd.RangeIndex(len(df)))

X_all, y_all = df.drop(columns="Class"), df["Class"].to_numpy()
split = np.load(SPLIT_PATH)
train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]

_trval, _test = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y_all, random_state=SPLIT_SEED)
_train, _val = train_test_split(_trval, test_size=0.25, stratify=y_all[_trval], random_state=SPLIT_SEED)
assert np.array_equal(_train, train_idx) and np.array_equal(_val, val_idx) and np.array_equal(_test, test_idx), "split_indices.npz diferă de splitul oficial"

X_train, y_train = X_all.loc[train_idx], y_all[train_idx]
X_val, y_val = X_all.loc[val_idx], y_all[val_idx]
assert (int(y_train.sum()), int(y_val.sum())) == (295, 99) and len(val_idx) == 56962
del X_all, y_all, df, _trval, _test, _train, _val, test_idx   # testul nu se folosește în Exp. 8

print(f"train {len(train_idx):,} ({int(y_train.sum())} fraude)   validation {len(val_idx):,} ({int(y_val.sum())} fraude)   test: neatins")

## 3. Circuitul Exp. 3 (copiat neschimbat) și funcțiile oficiale

`feature_map`, `ansatz`, `vqc`, `undersample`, `AngleEncoder`, `choose_threshold`, `evaluate` sunt copiate din Exp. 3 / notebookul oficial. `noisy_qnode`, `noisy_predict` și `noise_locations` primesc un singur argument nou, `gate_set`, cu valoarea implicită `NATIVE_GATES`: cu valoarea implicită sunt **identice** cu cele oficiale (Scenariul A); Scenariul B le apelează cu `ELEMENTARY_GATES`, pentru ambele circuite.

In [ ]:
def undersample(X_part, y_part, seed):
    rng = np.random.default_rng(seed)
    fraud = np.flatnonzero(y_part == 1)
    legit = rng.choice(np.flatnonzero(y_part == 0), size=len(fraud), replace=False)
    keep = rng.permutation(np.concatenate([fraud, legit]))
    return X_part.iloc[keep], y_part[keep]


class AngleEncoder:
    raw_columns = ["Time", "Amount"]

    def __init__(self, n_components, seed):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components, random_state=seed)
        self.angles = MinMaxScaler(feature_range=(0, np.pi))

    def _scaled(self, frame):
        out = frame.copy()
        out[self.raw_columns] = self.scaler.transform(frame[self.raw_columns])
        return out.to_numpy()

    def fit(self, frame):
        self.scaler.fit(frame[self.raw_columns])
        self.angles.fit(self.pca.fit_transform(self._scaled(frame)))
        return self

    def transform(self, frame):
        return np.clip(self.angles.transform(self.pca.transform(self._scaled(frame))), 0, np.pi)


def feature_map(x):
    """ZZ feature map. x holds N_QUBITS angles in [0, pi], or a batch of them (shape (..., N_QUBITS))."""
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[..., i], wires=i)
        for i in range(N_QUBITS):
            for j in range(i + 1, N_QUBITS):
                qml.CNOT(wires=[i, j])
                qml.RZ(2 * (np.pi - x[..., i]) * (np.pi - x[..., j]), wires=j)
                qml.CNOT(wires=[i, j])


def ansatz(weights):
    """EfficientSU2-style ansatz; weights has shape (ANSATZ_REPS + 1, N_QUBITS, 2)."""
    for layer in range(ANSATZ_REPS + 1):
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        if layer < ANSATZ_REPS:
            for i in range(N_QUBITS - 1):
                qml.CNOT(wires=[i, i + 1])


def vqc(x, weights):
    feature_map(x)
    ansatz(weights)
    return qml.expval(qml.PauliZ(0))


dev4 = qml.device("default.qubit", wires=N_QUBITS)   # shots=None: valori de așteptare exacte


def readout(expval):
    return (1 - expval) / 2


def clean_predict(Z, weights, circuit_fn, chunk=20000):
    """Fraud scores without noise (default.qubit), as `predict` in Exp. 3."""
    qnode = qml.QNode(circuit_fn, dev4, interface="torch")
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([readout(qnode(X_t[k:k + chunk], weights)) for k in range(0, len(X_t), chunk)]).numpy()


def choose_threshold(y_true, scores):
    """The threshold that maximises F1 on (y_true, scores)."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return thresholds[np.argmax(f1[:-1])]


def evaluate(y_true, scores, threshold):
    predicted = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "AUPRC": average_precision_score(y_true, scores),
        "AUROC": roc_auc_score(y_true, scores),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "recall": recall_score(y_true, predicted, zero_division=0),
        "F1": f1_score(y_true, predicted, zero_division=0),
        "MCC": matthews_corrcoef(y_true, predicted),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    }


def noisy_qnode(circuit_fn, p, n_wires=N_QUBITS, gate_set=NATIVE_GATES):
    """QNode of circuit_fn under the challenge's noise model: decomposition into gate_set, then a depolarizing
    channel of probability p after every gate, on each of its qubits."""
    noisy_device = qml.noise.insert(qml.device("default.mixed", wires=n_wires), qml.DepolarizingChannel, p, position="all")
    qnode = qml.QNode(circuit_fn, noisy_device, interface="torch", diff_method="backprop")
    return qml.transforms.decompose(qnode, gate_set=gate_set)


def noisy_predict(Z, weights, p, circuit_fn=vqc, gate_set=NATIVE_GATES, n_wires=N_QUBITS, chunk=2000):
    """Fraud scores of the transactions Z when circuit_fn runs under depolarizing noise p."""
    noisy_circuit = noisy_qnode(circuit_fn, p, n_wires, gate_set)
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([readout(noisy_circuit(X_t[k:k + chunk], weights)) for k in range(0, len(X_t), chunk)]).numpy()


def noise_locations(circuit_fn, *args, n_wires=N_QUBITS, gate_set=NATIVE_GATES):
    """Number of depolarizing channels the noise model inserts: one per qubit of every gate after decomposition."""
    qnode = qml.transforms.decompose(qml.QNode(circuit_fn, qml.device("default.qubit", wires=n_wires)), gate_set=gate_set)
    return sum(size * count for size, count in qml.specs(qnode)(*args)["resources"].gate_sizes.items())

## 4. Încărcarea modelelor Exp. 3 (fără reantrenare)

Pentru fiecare seed: `undersample` 1:1 → `AngleEncoder(4, seed)` fitat pe setul 1:1 (identic cu Exp. 3) → ponderile salvate în `results/result_exp3/weights/seed_<s>.pt`. Verificăm că ponderile reproduc AUPRC-ul salvat la p = 0.

In [ ]:
def load_jsonl(path, required=False):
    """Înregistrările din path. Un fișier obligatoriu lipsă oprește execuția, în loc să întoarcă o listă goală."""
    if not path.exists():
        if required:
            raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def to_builtin(value):
    if isinstance(value, dict):
        return {k: to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(to_builtin(record), ensure_ascii=False) + "\n")


models = {}
for seed in SEEDS:
    X_s, y_s = undersample(X_train, y_train, seed)
    encoder = AngleEncoder(N_QUBITS, seed).fit(X_s)
    saved = torch.load(EXP3_DIR / "weights" / f"seed_{seed}.pt", weights_only=True)
    weights = saved["weights"].detach().clone()
    assert weights.shape == (ANSATZ_REPS + 1, N_QUBITS, 2) and weights.dtype == torch.float64
    Z_val = encoder.transform(X_val)
    t0 = time.perf_counter()
    clean_exp3 = clean_predict(Z_val, weights, vqc)
    t_clean = time.perf_counter() - t0
    ap = average_precision_score(y_val, clean_exp3)
    assert np.isclose(ap, saved["best_val_ap"], rtol=0, atol=1e-12), f"seed {seed}: {ap} vs {saved['best_val_ap']}"
    models[seed] = {"weights": weights, "Z_val": Z_val, "Z_tr": encoder.transform(X_s), "y_tr": y_s,
                    "clean": {"exp3": clean_exp3}, "clean_time": {"exp3": t_clean}}
    print(f"seed {seed}: ponderi Exp. 3 încărcate   val AUPRC (p = 0) {ap:.6f} = salvat {saved['best_val_ap']:.6f}")
print("✅ 5 modele Exp. 3 reconstruite, fără reantrenare")

## 5. Inspecția circuitului: unde se poate fuziona?

Nu presupunem nimic: parcurgem circuitul Exp. 3 **după** decompoziția oficială în `NATIVE_GATES` și găsim, pe fiecare fir, **secvențele maximale de porți de un qubit consecutive** (fără nicio poartă de doi qubiți pe acel fir între ele). Doar o secvență de lungime ≥ 2 poate deveni o singură poartă fără să reordonăm nimic. Comparăm și cu cele două transformări PennyLane:
- `qml.transforms.merge_rotations`: unește doar rotații **de același tip** pe același fir (ex. RZ·RZ);
- `qml.transforms.single_qubit_fusion`: unește orice bloc de porți de un qubit într-un `Rot`.

In [ ]:
def tape_of(circuit_fn, x, weights, gate_set=NATIVE_GATES):
    """The tape of circuit_fn after decomposition into gate_set (numpy inputs: one transaction)."""
    qnode = qml.transforms.decompose(qml.QNode(circuit_fn, dev4), gate_set=gate_set)
    return qml.workflow.construct_tape(qnode)(x, weights)


def single_qubit_runs(ops):
    """Maximal runs of consecutive single-qubit gates on each wire (no multi-qubit gate on that wire in between),
    sorted by wire and then by position, plus the ordered list of multi-qubit gates (the skeleton)."""
    runs, open_runs, skeleton = [], {}, []
    for k, op in enumerate(ops):
        if len(op.wires) == 1:
            open_runs.setdefault(op.wires[0], []).append((k, op))
        else:
            skeleton.append((op.name, tuple(op.wires.tolist())))
            for w in op.wires:
                if w in open_runs:
                    runs.append((w, open_runs.pop(w)))
    runs += list(open_runs.items())
    return sorted(runs, key=lambda r: (r[0], r[1][0][0])), skeleton


IDENTITIES = {
    ("Hadamard", "RZ"): ("Rot(π, π/2, 2·x_i)", "RZ(θ)·H = i·Rot(π, π/2, θ)   [H = i·RY(π/2)·RZ(π)]"),
    ("RY", "RZ"): ("Rot(0, a, b)", "RZ(b)·RY(a) = Rot(0, a, b)   [Rot(φ, θ, ω) = RZ(ω)·RY(θ)·RZ(φ)]"),
    ("RZ",): ("neschimbat", "poartă singură între două CNOT-uri: nimic de fuzionat"),
}

w_probe = models[SEEDS[0]]["weights"].numpy()
x_probe = models[SEEDS[0]]["Z_val"][0]
orig_ops = tape_of(vqc, x_probe, w_probe).operations
n_feature_map_ops = len(qml.workflow.construct_tape(qml.QNode(lambda x: (feature_map(x), qml.state())[1], dev4))(x_probe).operations)
assert len(orig_ops) == 67 and n_feature_map_ops == 26

runs, _ = single_qubit_runs(orig_ops)
position_on_wire = {}
fusion_rows = []
for wire, run in runs:
    names = tuple(op.name for _, op in run)
    assert names in IDENTITIES, f"secvență neașteptată pe firul {wire}: {names}"
    position_on_wire[wire] = position_on_wire.get(wire, 0) + 1
    fused_gate, identity = IDENTITIES[names]
    fusion_rows.append({
        "regiune": "feature map (ZZ)" if run[0][0] < n_feature_map_ops else "ansatz",
        "qubit": int(wire), "secvența nr. pe qubit": position_on_wire[wire], "index în circuit": run[0][0],
        "porți originale": " → ".join(names), "poartă după fuziune": fused_gate, "identitate": identity,
        "porți eliminate": len(names) - 1, "locuri de zgomot eliminate (Scenariul A)": len(names) - 1,
    })
fusion = pd.DataFrame(fusion_rows).sort_values(["index în circuit", "qubit"]).reset_index(drop=True)
fusion_summary = (fusion.groupby(["regiune", "porți originale", "poartă după fuziune"], sort=False)
                  .agg(blocuri=("qubit", "size"), **{"porți eliminate": ("porți eliminate", "sum")}).reset_index())
print(f"secvențe de porți de un qubit: {len(fusion)}   fuzionabile (lungime ≥ 2): {(fusion['porți eliminate'] > 0).sum()}   porți eliminate: {fusion['porți eliminate'].sum()}")
display(fusion_summary)

### Transformările automate PennyLane

Le aplicăm pe tape-ul circuitului original (intrări numpy: cu tensori torch, `single_qubit_fusion` din PennyLane 0.45.1 eșuează pentru că unghiurile lui H sunt numpy, iar cele ale RZ torch).

In [ ]:
transform_rows = []
for name, transform in [("merge_rotations", qml.transforms.merge_rotations), ("single_qubit_fusion", qml.transforms.single_qubit_fusion)]:
    (new_tape,), _ = transform(tape_of(vqc, x_probe, w_probe))   # tape nou pentru fiecare transformare
    counts = pd.Series([op.name for op in new_tape.operations]).value_counts().to_dict()
    transform_rows.append({"transformare": name, "operații după": len(new_tape.operations),
                           "operații eliminate": len(orig_ops) - len(new_tape.operations),
                           "tipuri": ", ".join(f"{k}: {v}" for k, v in sorted(counts.items())),
                           "CNOT păstrate": counts.get("CNOT", 0)})
    if name == "single_qubit_fusion":
        auto_fused_tape = new_tape
transforms = pd.DataFrame(transform_rows).set_index("transformare")
display(transforms)

# single_qubit_fusion transformă și cele 6 RZ izolate din blocurile ZZ în Rot (1 poartă → 1 poartă) și adaugă un GlobalPhase.
lone_rz = int((fusion["porți eliminate"] == 0).sum())
auto_counts = pd.Series([op.name for op in auto_fused_tape.operations]).value_counts()
assert auto_counts.get("Rot", 0) == (fusion["porți eliminate"] > 0).sum() + lone_rz
print(f"single_qubit_fusion: {auto_counts['Rot']} Rot = {(fusion['porți eliminate'] > 0).sum()} fuziuni reale + {lone_rz} RZ izolate rescrise ca Rot (fără câștig), "
      f"plus {auto_counts.get('GlobalPhase', 0)} GlobalPhase.")

**Decizie de implementare.** `merge_rotations` nu găsește nimic (nu există rotații adiacente de același tip). `single_qubit_fusion` găsește aceleași 20 de blocuri, dar (1) nu acceptă tensorii torch ai modelului, (2) rescrie și cele 6 RZ izolate ca `Rot` (fără câștig în Scenariul A, cu pierdere în Scenariul B, unde un `Rot` devine 3 porți) și (3) adaugă o operație `GlobalPhase`. Folosim deci o **fuziune manuală controlată**, care aplică exact cele două identități din tabel și lasă restul circuitului neatins; rezultatul automat rămâne o verificare încrucișată (secțiunea 6).

Identitățile (convenția PennyLane: `Rot(φ, θ, ω) = RZ(ω)·RY(θ)·RZ(φ)`):
1. **Ansatz:** `RY(a)` urmat de `RZ(b)` = `RZ(b)·RY(a)` = `Rot(0, a, b)`, **exact** (fără fază globală). Aceleași ponderi `a, b`, niciun parametru antrenabil nou (φ = 0 este o constantă).
2. **Feature map:** `H` urmat de `RZ(2x)`. Cum `RY(π/2)·Z = H` și `Z = i·RZ(π)`, avem `H = i·RY(π/2)·RZ(π)`, deci `RZ(2x)·H = i·Rot(π, π/2, 2x)`: egal până la faza globală `i`, care nu schimbă nicio stare fizică (ρ → UρU†) și niciun expval.

In [ ]:
def fused_feature_map(x):
    """ZZ feature map of Exp. 3 with every H, RZ(2x) pair fused: RZ(2x)·H = i·Rot(π, π/2, 2x)."""
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            angle = 2 * x[..., i]
            ones = qml.math.ones_like(angle)
            qml.Rot(np.pi * ones, np.pi / 2 * ones, angle, wires=i)
        for i in range(N_QUBITS):   # blocurile ZZ, neschimbate: RZ-ul dintre cele două CNOT-uri nu are cu cine fuziona
            for j in range(i + 1, N_QUBITS):
                qml.CNOT(wires=[i, j])
                qml.RZ(2 * (np.pi - x[..., i]) * (np.pi - x[..., j]), wires=j)
                qml.CNOT(wires=[i, j])


def fused_ansatz(weights):
    """Ansatz of Exp. 3 with every RY(a), RZ(b) pair fused: RZ(b)·RY(a) = Rot(0, a, b), same weights."""
    for layer in range(ANSATZ_REPS + 1):
        for i in range(N_QUBITS):
            qml.Rot(qml.math.zeros_like(weights[layer, i, 0]), weights[layer, i, 0], weights[layer, i, 1], wires=i)
        if layer < ANSATZ_REPS:
            for i in range(N_QUBITS - 1):
                qml.CNOT(wires=[i, i + 1])


def fused_vqc(x, weights):
    fused_feature_map(x)
    fused_ansatz(weights)
    return qml.expval(qml.PauliZ(0))


CIRCUITS = {"exp3": vqc, "exp8": fused_vqc}

fused_ops = tape_of(fused_vqc, x_probe, w_probe).operations
_, skeleton_orig = single_qubit_runs(orig_ops)
fused_runs, skeleton_fused = single_qubit_runs(fused_ops)
assert skeleton_orig == skeleton_fused, "CNOT-urile trebuie să rămână identice, în aceeași ordine"
assert all(len(run) == 1 for _, run in fused_runs), "după fuziune nu mai rămân porți de un qubit consecutive"
assert len(fused_ops) == len(orig_ops) - fusion["porți eliminate"].sum()
print(f"✅ circuit fuzionat: {len(orig_ops)} → {len(fused_ops)} porți, aceleași {len(skeleton_orig)} CNOT în aceeași ordine")

## 6. Verificarea echivalenței fără zgomot (etapa critică)

Pentru fiecare seed, cu ponderile originale:
1. **Per bloc:** matricea 2×2 a fiecărei secvențe originale vs matricea `Rot`-ului care o înlocuiește, aliniate la faza globală optimă (`max |U − e^{iα}V|`); pe 100 de tranzacții de validation (50 fraude + 50 legitime, alese cu `rng(seed)`) și 100 de vectori aleatori de unghiuri din [0, π].
2. **Circuitul întreg:** unitara 16×16 (până la fază globală), pe aceleași intrări.
3. **⟨Z₀⟩ și scorul de fraudă** pe **toată** validarea (56 962 tranzacții) și pe 5 000 de unghiuri aleatoare din [0, π]: eroare maximă și medie absolută.
4. Cross-check: unitara obținută cu `single_qubit_fusion` = unitara fuziunii manuale.
5. `default.mixed` la p = 0 (Scenariile A și B) = `default.qubit`.

Toleranță: 1e-10 (float64). Dacă o verificare pică, evaluarea sub zgomot nu se rulează.

In [ ]:
def run_unitary(run):
    U = np.eye(2, dtype=complex)
    for _, op in run:
        U = qml.matrix(op) @ U
    return U


def phase_aligned_error(U, V):
    """max |U − e^{iα}V| for the best global phase α, and α."""
    alpha = np.angle(np.trace(V.conj().T @ U))
    return np.abs(U - np.exp(1j * alpha) * V).max(), alpha


def check_inputs(seed):
    rng = np.random.default_rng(seed)
    m = models[seed]
    rows = np.concatenate([rng.choice(np.flatnonzero(y_val == 1), N_VAL_UNITARY // 2, replace=False),
                           rng.choice(np.flatnonzero(y_val == 0), N_VAL_UNITARY // 2, replace=False)])
    return {"validation": m["Z_val"][rows], "aleator [0, π]": rng.uniform(0, np.pi, (N_VAL_UNITARY, N_QUBITS))}


matrix_of = {c: qml.matrix(qml.QNode(fn, dev4), wire_order=range(N_QUBITS)) for c, fn in CIRCUITS.items()}
block_records, check_rows = [], []
for seed in SEEDS:
    m = models[seed]
    w = m["weights"].numpy()
    for source, X_check in check_inputs(seed).items():
        full_errors = []
        for x in X_check:
            runs_o, skel_o = single_qubit_runs(tape_of(vqc, x, w).operations)
            runs_f, skel_f = single_qubit_runs(tape_of(fused_vqc, x, w).operations)
            assert skel_o == skel_f and len(runs_o) == len(runs_f)
            for (wire_o, run_o), (wire_f, run_f) in zip(runs_o, runs_f):
                assert wire_o == wire_f and len(run_f) == 1
                if len(run_o) > 1:
                    err, alpha = phase_aligned_error(run_unitary(run_o), run_unitary(run_f))
                    block_records.append({"seed": seed, "intrări": source, "bloc": " → ".join(op.name for _, op in run_o) + " ⇒ Rot",
                                          "eroare": err, "fază globală α / π": alpha / np.pi})
            full_errors.append(phase_aligned_error(matrix_of["exp3"](x, w), matrix_of["exp8"](x, w))[0])
        check_rows.append({"seed": seed, "verificare": "unitara 16×16 a circuitului (până la fază globală)", "intrări": source,
                           "comparații": len(full_errors), "eroare max": max(full_errors), "eroare medie": float(np.mean(full_errors))})

    # expval și scor pe toată validarea și pe unghiuri aleatoare
    m["clean"]["exp8"] = clean_predict(m["Z_val"], m["weights"], fused_vqc)
    t0 = time.perf_counter()
    clean_predict(m["Z_val"], m["weights"], fused_vqc)
    m["clean_time"]["exp8"] = time.perf_counter() - t0
    X_random = np.random.default_rng(1000 + seed).uniform(0, np.pi, (N_RANDOM, N_QUBITS))
    for source, s_o, s_f in [("validation (toate 56 962)", m["clean"]["exp3"], m["clean"]["exp8"]),
                             (f"aleator [0, π] ({N_RANDOM})", clean_predict(X_random, m["weights"], vqc), clean_predict(X_random, m["weights"], fused_vqc))]:
        d_score = np.abs(s_o - s_f)
        check_rows.append({"seed": seed, "verificare": "⟨Z₀⟩ (diferență absolută)", "intrări": source,
                           "comparații": len(d_score), "eroare max": 2 * d_score.max(), "eroare medie": 2 * d_score.mean()})
        check_rows.append({"seed": seed, "verificare": "scor de fraudă (1 − ⟨Z₀⟩)/2", "intrări": source,
                           "comparații": len(d_score), "eroare max": d_score.max(), "eroare medie": d_score.mean()})
    # AUPRC identic (rangurile pot diferi doar la egalități de ordinul 1e-16)
    check_rows.append({"seed": seed, "verificare": "|Δ AUPRC| fără zgomot", "intrări": "validation (toate 56 962)", "comparații": 1,
                       "eroare max": abs(average_precision_score(y_val, m["clean"]["exp3"]) - average_precision_score(y_val, m["clean"]["exp8"])),
                       "eroare medie": np.nan})

blocks = pd.DataFrame(block_records)
for (seed, source, block), group in blocks.groupby(["seed", "intrări", "bloc"], sort=False):
    check_rows.append({"seed": seed, "verificare": f"unitara per bloc: {block}", "intrări": source, "comparații": len(group),
                       "eroare max": group["eroare"].max(), "eroare medie": group["eroare"].mean()})
print("faza globală observată per tip de bloc (α / π):")
display(blocks.groupby("bloc")["fază globală α / π"].agg(["min", "max"]))

# cross-check cu single_qubit_fusion (intrări numpy, seed 0)
auto_errors = []
for x in check_inputs(SEEDS[0])["validation"][:20]:
    (auto_tape,), _ = qml.transforms.single_qubit_fusion(tape_of(vqc, x, w_probe))
    auto_errors.append(phase_aligned_error(qml.matrix(auto_tape, wire_order=range(N_QUBITS)), matrix_of["exp8"](x, w_probe))[0])
check_rows.append({"seed": SEEDS[0], "verificare": "single_qubit_fusion vs fuziunea manuală (unitara 16×16)", "intrări": "validation",
                   "comparații": len(auto_errors), "eroare max": max(auto_errors), "eroare medie": float(np.mean(auto_errors))})

# default.mixed la p = 0, ambele scenarii, vs default.qubit (seed 0, 2000 tranzacții)
m0 = models[SEEDS[0]]
for scenario, gate_set in GATE_SETS.items():
    for c, fn in CIRCUITS.items():
        d = np.abs(noisy_predict(m0["Z_val"][:2000], m0["weights"], 0.0, fn, gate_set) - m0["clean"]["exp3"][:2000])
        check_rows.append({"seed": SEEDS[0], "verificare": f"default.mixed p = 0, Scenariul {scenario}, {CIRCUIT_LABELS[c]} vs default.qubit Exp. 3",
                           "intrări": "validation (2000)", "comparații": len(d), "eroare max": d.max(), "eroare medie": d.mean()})

equivalence = pd.DataFrame(check_rows)
equivalence["toleranță"] = EQUIV_TOL
equivalence["trecut"] = equivalence["eroare max"] <= EQUIV_TOL
summary_eq = equivalence.groupby(["verificare", "intrări"], sort=False).agg(
    seeds=("seed", "nunique"), comparații=("comparații", "sum"), **{"eroare max": ("eroare max", "max")},
    **{"eroare medie": ("eroare medie", "mean")}, trecut=("trecut", "all"))
display(summary_eq)
EQUIVALENCE_OK = bool(equivalence["trecut"].all())
assert EQUIVALENCE_OK, "circuitul fuzionat NU este echivalent fără zgomot: nu continuăm"
print(f"✅ echivalență fără zgomot: toate cele {len(equivalence)} verificări ≤ {EQUIV_TOL:.0e} (eroare maximă globală {equivalence['eroare max'].max():.1e})")

### 6b. Precizia simulatorului `default.mixed` (diagnostic, nu ține de fuziune)

La prima rulare, verificarea „`default.mixed` la p = 0 = `default.qubit`” a dat ~7e-8 **și pentru circuitul original Exp. 3**. Cauza: cu dtype-ul implicit torch `float32` (setarea din Exp. 3 / Exp. 6), `default.mixed` construiește unele constante în precizie simplă, chiar și fără canale de zgomot. Cu `float64` implicit, eroarea scade la ~1e-13. De aceea Exp. 8 rulează **ambele** circuite cu `float64` implicit (condiții numerice comparabile), iar reproducerea scorurilor istorice Exp. 3 / Exp. 6 se verifică cu toleranța `REPRO_TOL = 1e-6`, justificată de tabelul de mai jos.

In [ ]:
precision_rows = []
Z_prec, w_prec = models[SEEDS[0]]["Z_val"][:2000], models[SEEDS[0]]["weights"]
for dtype in [torch.float32, torch.float64]:
    torch.set_default_dtype(dtype)
    for scenario, gate_set in GATE_SETS.items():
        for c, fn in CIRCUITS.items():
            d = np.abs(noisy_predict(Z_prec, w_prec, 0.0, fn, gate_set) - models[SEEDS[0]]["clean"]["exp3"][:2000])
            precision_rows.append({"dtype implicit torch": str(dtype), "scenariu": scenario, "circuit": CIRCUIT_LABELS[c],
                                   "eroare max vs default.qubit": d.max(), "eroare medie": d.mean()})
torch.set_default_dtype(TORCH_DEFAULT_DTYPE)
simulator_precision = pd.DataFrame(precision_rows).set_index(["dtype implicit torch", "scenariu", "circuit"])
display(simulator_precision)
assert simulator_precision.loc[str(torch.float64), "eroare max vs default.qubit"].max() < EQUIV_TOL
assert simulator_precision["eroare max vs default.qubit"].max() < REPRO_TOL
print(f"dtype implicit torch restaurat: {torch.get_default_dtype()}")

## 7. Structura circuitelor în cele două scenarii

- **Scenariul A (oficial):** decompoziție în `NATIVE_GATES` (Rot și H native), un canal depolarizant pe fiecare qubit al fiecărei porți.
- **Scenariul B (control):** decompoziție în `{RX, RY, RZ, CNOT}` cu regulile implicite PennyLane, pentru **ambele** circuite: `Rot(φ, θ, ω) → RZ(φ)·RY(θ)·RZ(ω)` (inclusiv `RZ(0)` când φ = 0, cum ar face un compilator care tratează `Rot` ca poartă generică), `H → RZ(π/2)·RX(π/2)·RZ(π/2)`. Același model de zgomot după decompoziție.
- **B′ (doar structural, neevaluat sub zgomot):** Scenariul B după eliminarea rotațiilor de unghi exact 0 (ce ar face un compilator care vede valorile).

Adâncimea este calculată din tape și verificată față de `qml.specs`; locurile de zgomot față de `noise_locations` oficial. Execuțiile parameter-shift per exemplu sunt **măsurate** cu `qml.Tracker` (mini-batch de 32, dispozitiv cu 1024 shots).

In [ ]:
def circuit_depth(ops):
    depth = {}
    for op in ops:
        level = 1 + max(depth.get(w, 0) for w in op.wires)
        for w in op.wires:
            depth[w] = level
    return max(depth.values())


def is_zero_rotation(op):
    return op.name in {"RX", "RY", "RZ"} and abs(float(op.data[0])) < 1e-12


def structure_row(circuit, scenario, ops, prune_zero=False):
    if prune_zero:
        ops = [op for op in ops if not is_zero_rotation(op)]
    names = pd.Series([op.name for op in ops]).value_counts()
    one_q = sum(len(op.wires) == 1 for op in ops)
    two_q = sum(len(op.wires) == 2 for op in ops)
    return {"scenariu": scenario, "circuit": circuit, "porți native": len(ops), "porți 1 qubit": one_q, "porți 2 qubiți": two_q,
            "CNOT": int(names.get("CNOT", 0)), "adâncime": circuit_depth(ops), "locuri de zgomot": one_q + 2 * two_q,
            "rotații de unghi 0": sum(is_zero_rotation(op) for op in ops),
            **{f"#{g}": int(names.get(g, 0)) for g in ["Hadamard", "Rot", "RX", "RY", "RZ", "CNOT"]}}


x_t, w_t = torch.tensor(x_probe), models[SEEDS[0]]["weights"]
structure_rows = []
for scenario, gate_set in GATE_SETS.items():
    for c, fn in CIRCUITS.items():
        ops = tape_of(fn, x_probe, w_probe, gate_set).operations
        row = structure_row(c, scenario, ops)
        specs = qml.specs(qml.transforms.decompose(qml.QNode(fn, dev4), gate_set=gate_set))(x_t, w_t)["resources"]
        assert row["adâncime"] == specs.depth and row["porți native"] == sum(specs.gate_types.values())
        assert row["locuri de zgomot"] == noise_locations(fn, x_t, w_t, gate_set=gate_set)
        assert all(op.name in gate_set for op in ops)
        structure_rows.append(row)
        if scenario == "B":
            structure_rows.append(structure_row(c, "B′", ops, prune_zero=True))

# execuții parameter-shift măsurate (hardware-like, 1024 shots), mini-batch de 32 din setul 1:1 al seed-ului 0
batch_x = torch.tensor(models[SEEDS[0]]["Z_tr"][:BATCH_SIZE])
batch_y = torch.tensor(models[SEEDS[0]]["y_tr"][:BATCH_SIZE], dtype=torch.float64)
tracker_per_example = {}
for c, fn in CIRCUITS.items():
    shot_device = qml.device("default.qubit", wires=N_QUBITS, seed=SPLIT_SEED)
    hardware_like = qml.set_shots(qml.QNode(fn, shot_device, interface="torch", diff_method="parameter-shift"), shots=SHOTS_PER_EXECUTION)
    w_probe_param = torch.nn.Parameter(models[SEEDS[0]]["weights"].clone())
    with qml.Tracker(shot_device) as tracker:
        loss = torch.nn.BCELoss()(readout(hardware_like(batch_x, w_probe_param)), batch_y)
        loss.backward()
    tracker_per_example[c] = tracker.totals["executions"] / len(batch_x)
    assert w_probe_param.numel() == 32

circuit_comparison = pd.DataFrame(structure_rows)
circuit_comparison["parametri antrenabili"] = 32
circuit_comparison["execuții / exemplu (nominal 1 + 2P)"] = 1 + 2 * 32
circuit_comparison["execuții / exemplu (Tracker)"] = circuit_comparison["circuit"].map(tracker_per_example)
circuit_comparison["circuit"] = circuit_comparison["circuit"].map(CIRCUIT_LABELS)
circuit_comparison = circuit_comparison.set_index(["scenariu", "circuit"])
exp3_ref = pd.read_csv(EXP3_DIR / "circuit_structure.csv", index_col=0).loc["Exp. 3 (ZZ ×1)"]
a_orig = circuit_comparison.loc[("A", ORIG_LABEL)]
assert (a_orig["porți native"], a_orig["CNOT"], a_orig["adâncime"], a_orig["locuri de zgomot"]) == (
    exp3_ref["porți native"], exp3_ref["CNOT"], exp3_ref["adâncime"], exp3_ref["locuri de zgomot"]), "Exp. 3 reconstruit ≠ structura salvată"
STRUCT = circuit_comparison
print("✅ Scenariul A, Exp. 3 reconstruit = structura salvată (67 porți, 21 CNOT, adâncime 31, 88 locuri de zgomot)")
display(circuit_comparison)

## 8. Configurația și directorul de rezultate

In [ ]:
config = {
    "experiment": EXPERIMENT_NAME,
    "base_model": "Exp. 3 (FEATURE_MAP_REPS=1, ANSATZ_REPS=3, 4 qubits, readout PauliZ(0)), weights from results/result_exp3/weights, no retraining",
    "change": "single-qubit gate fusion only: H·RZ -> Rot (feature map), RY·RZ -> Rot (ansatz)",
    "fusion_identities": {k[0] + "," + k[1]: v[1] for k, v in IDENTITIES.items() if len(k) == 2},
    "fusion_method": "manual, controlled (single_qubit_fusion used only as cross-check)",
    "seeds": SEEDS, "noise_levels": P_ALL,
    "scenarios": {s: {"label": SCENARIO_LABELS[s], "gate_set": sorted(g)} for s, g in GATE_SETS.items()},
    "scenario_B_rules": "PennyLane default decompositions: Rot(φ,θ,ω) -> RZ(φ) RY(θ) RZ(ω); Hadamard -> RZ(π/2) RX(π/2) RZ(π/2)",
    "noise_model": "qml.noise.insert(default.mixed, DepolarizingChannel, p, position='all') after decomposition into the scenario's gate set",
    "thresholds_primary": "Exp. 3 validation thresholds per (seed, p), from results/result_exp3/val_runs.jsonl",
    "thresholds_secondary": "re-optimised max-F1 on the same validation set (optimistic)",
    "evaluation_set": "validation (test not used)",
    "torch_default_dtype": str(TORCH_DEFAULT_DTYPE), "reproduction_tolerance_vs_exp3_float32": REPRO_TOL,
    "equivalence_tolerance": EQUIV_TOL, "n_val_unitary": N_VAL_UNITARY, "n_random": N_RANDOM, "smoke_rows": SMOKE_ROWS,
    "shots_per_execution": SHOTS_PER_EXECUTION,
    "versions": {"pennylane": qml.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__, "numpy": np.__version__},
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RESULTS_DIR / "config.json"
VAL_RUNS_PATH = RESULTS_DIR / "val_runs.jsonl"
if CONFIG_PATH.exists():
    previous = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    current = json.loads(json.dumps(to_builtin(config)))
    if {k: v for k, v in previous.items() if k != "versions"} != {k: v for k, v in current.items() if k != "versions"}:
        raise RuntimeError(f"{CONFIG_PATH} conține altă configurație. Nu suprascriu: mută directorul sau schimbă RESULTS_DIR.")
else:
    CONFIG_PATH.write_text(json.dumps(to_builtin(config), indent=2, ensure_ascii=False), encoding="utf-8")

fusion.to_csv(RESULTS_DIR / "fusion_analysis.csv", index=False)
transforms.to_csv(RESULTS_DIR / "pennylane_transforms.csv")
equivalence.to_csv(RESULTS_DIR / "equivalence_checks.csv", index=False)
circuit_comparison.to_csv(RESULTS_DIR / "circuit_comparison.csv")
simulator_precision.to_csv(RESULTS_DIR / "simulator_precision.csv")
print("salvat: config.json, fusion_analysis.csv, pennylane_transforms.csv, equivalence_checks.csv, circuit_comparison.csv, simulator_precision.csv")

## 9. Test de fum (seed 0, primele 2 000 de tranzacții, p = 0.05)

Verifică pe un subset mic că (1) Exp. 3 reconstruit în Scenariul A reproduce scorurile salvate de Exp. 6 (aceleași ponderi, același model de zgomot) și (2) toate cele patru combinații rulează. Estimează și timpul evaluării complete.

In [ ]:
assert EQUIVALENCE_OK
m0 = models[SEEDS[0]]
Z_smoke = m0["Z_val"][:SMOKE_ROWS]
smoke_rows = []
for scenario, gate_set in GATE_SETS.items():
    for c, fn in CIRCUITS.items():
        t0 = time.perf_counter()
        s = noisy_predict(Z_smoke, m0["weights"], 0.05, fn, gate_set)
        elapsed = time.perf_counter() - t0
        smoke_rows.append({"scenariu": scenario, "circuit": CIRCUIT_LABELS[c], "timp [s]": elapsed,
                           "MAE față de fără zgomot": np.abs(s - m0["clean"][c][:SMOKE_ROWS]).mean(), "scores": s})
smoke = pd.DataFrame(smoke_rows)
cached = np.load(EXP6_SCORES_DIR / f"B_seed{SEEDS[0]}_p0.05.npz")["val"][:SMOKE_ROWS]
smoke_repro = np.abs(smoke.loc[(smoke["scenariu"] == "A") & (smoke["circuit"] == ORIG_LABEL), "scores"].iloc[0] - cached).max()
assert smoke_repro < REPRO_TOL, f"Exp. 3 reconstruit nu reproduce scorurile Exp. 6: {smoke_repro}"
estimate_min = smoke["timp [s]"].sum() * len(y_val) / SMOKE_ROWS * len(SEEDS) * len(NOISE_LEVELS) / 60
display(smoke.drop(columns="scores").set_index(["scenariu", "circuit"]))
print(f"✅ Exp. 3 reconstruit (A) vs scorurile Exp. 6, p = 0.05: diferență maximă {smoke_repro:.1e}")
print(f"estimare evaluare completă (5 seeds × 3 p × 4 combinații pe 56 962 tranzacții): ~{estimate_min:.0f} min")

## 10. Evaluarea completă pe validation

Pentru fiecare seed, scenariu și p: scorurile ambelor circuite (aceleași intrări, aceleași ponderi). La p = 0 scorurile sunt cele fără zgomot (identice în A și B). Fiecare (scenariu, seed, p) terminat se salvează imediat în `val_runs.jsonl`: o rulare întreruptă se reia de unde a rămas.

Metrici pe înregistrare: metricile clasificatorului la **pragul Exp. 3** pentru (seed, p) (comparația principală), aceleași metrici la pragul F1-max reoptimizat (secundar), Spearman ρ, MAE / RMSE ale scorurilor față de scorurile fără zgomot, diferența medie a ⟨Z₀⟩, timpul de simulare. Pentru Exp. 3 în Scenariul A se verifică și reproducerea rezultatelor salvate.

In [ ]:
assert EQUIVALENCE_OK
exp3_val = {(r["seed"], r["p"]): r for r in load_jsonl(EXP3_DIR / "val_runs.jsonl", required=True)}
TIE_COLS = ["precision", "recall", "F1", "MCC", "TP", "FP", "FN", "TN"]


def run_record(scenario, c, seed, p, scores, elapsed):
    s, s0 = scores[c], models[seed]["clean"][c]
    ref = exp3_val[(seed, p)]
    tuned_threshold = choose_threshold(y_val, s)
    tuned = evaluate(y_val, s, tuned_threshold)
    record = {"scenariu": scenario, "circuit": c, "seed": seed, "p": p, **evaluate(y_val, s, ref["threshold"]),
              "prag (Exp. 3)": ref["threshold"], "Spearman rho": spearmanr(s0, s).statistic,
              "MAE scor": np.abs(s - s0).mean(), "RMSE scor": float(np.sqrt(np.mean((s - s0) ** 2))), "max |Δ scor|": np.abs(s - s0).max(),
              "Δ ⟨Z0⟩ medie": float(np.mean(-2 * (s - s0))), "|Δ ⟨Z0⟩| medie": float(np.mean(2 * np.abs(s - s0))),
              "prag reoptimizat": tuned_threshold, **{f"{k} [prag reoptimizat]": tuned[k] for k in TIE_COLS},
              "timp scorare [s]": elapsed[c], "locuri de zgomot": int(STRUCT.loc[(scenario, CIRCUIT_LABELS[c]), "locuri de zgomot"])}
    if c == "exp8":
        record["MAE față de Exp. 3 (același p)"] = np.abs(s - scores["exp3"]).mean()
    if c == "exp3" and scenario == "A":
        cache = EXP6_SCORES_DIR / f"B_seed{seed}_p{p}.npz"
        record["|Δ AUPRC| vs Exp. 3 salvat"] = abs(record["AUPRC"] - ref["AUPRC"])
        record["|Δ prag F1-max| vs Exp. 3 salvat"] = abs(tuned_threshold - ref["threshold"])
        record["|Δ F1| vs Exp. 3 salvat"] = abs(record["F1"] - ref["F1"])
        record["max |Δ scor| vs cache Exp. 6"] = np.abs(s - np.load(cache)["val"]).max() if cache.exists() else None
    return record


done = {(r["scenariu"], r["circuit"], r["seed"], r["p"]) for r in load_jsonl(VAL_RUNS_PATH)}
todo = [(sc, seed, p) for seed in SEEDS for sc in GATE_SETS for p in P_ALL if not all((sc, c, seed, p) in done for c in CIRCUITS)]
print(f"(scenariu, seed, p) de evaluat: {len(todo)} din {len(SEEDS) * len(GATE_SETS) * len(P_ALL)}")
for scenario, seed, p in todo:
    m = models[seed]
    scores, elapsed = {}, {}
    for c, fn in CIRCUITS.items():
        if p == 0.0:
            scores[c], elapsed[c] = m["clean"][c], m["clean_time"][c]
        else:
            t0 = time.perf_counter()
            scores[c] = noisy_predict(m["Z_val"], m["weights"], p, fn, GATE_SETS[scenario])
            elapsed[c] = time.perf_counter() - t0
    records = [run_record(scenario, c, seed, p, scores, elapsed) for c in CIRCUITS]
    for record in records:
        append_jsonl(VAL_RUNS_PATH, record)
    o, f = records
    print(f"{scenario}  seed {seed}  p = {p:<5}  AUPRC {o['AUPRC']:.4f} → {f['AUPRC']:.4f}   MAE {o['MAE scor']:.5f} → {f['MAE scor']:.5f}   "
          f"rho {o['Spearman rho']:.4f} → {f['Spearman rho']:.4f}   ({elapsed['exp3']:.0f} s / {elapsed['exp8']:.0f} s)")

### Reproducerea Exp. 3 și integritatea rezultatelor

In [ ]:
runs = pd.DataFrame(load_jsonl(VAL_RUNS_PATH, required=True))
keys = list(zip(runs["scenariu"], runs["circuit"], runs["seed"], runs["p"]))
assert len(keys) == len(set(keys)), "rezultate duplicate în val_runs.jsonl"
missing = {(sc, c, s, p) for sc in GATE_SETS for c in CIRCUITS for s in SEEDS for p in P_ALL} - set(keys)
assert not missing, f"lipsesc: {sorted(missing)}"
runs = runs.sort_values(["scenariu", "circuit", "p", "seed"]).reset_index(drop=True)

repro_cols = ["|Δ AUPRC| vs Exp. 3 salvat", "|Δ prag F1-max| vs Exp. 3 salvat", "|Δ F1| vs Exp. 3 salvat", "max |Δ scor| vs cache Exp. 6"]
repro = runs[(runs["scenariu"] == "A") & (runs["circuit"] == "exp3")].astype({col: float for col in repro_cols})
print("Reproducerea Exp. 3 (Scenariul A), maxim pe seeds, per p:")
display(repro.groupby("p")[repro_cols].max())
assert repro["|Δ AUPRC| vs Exp. 3 salvat"].max() < REPRO_TOL and repro["|Δ prag F1-max| vs Exp. 3 salvat"].max() < REPRO_TOL
assert repro["max |Δ scor| vs cache Exp. 6"].dropna().max() < REPRO_TOL
n_f1_changed = int((repro["|Δ F1| vs Exp. 3 salvat"] > 0).sum())
print(f"✅ Exp. 3 reconstruit (float64) reproduce AUPRC, pragurile și scorurile istorice (float32) în {REPRO_TOL:.0e}; "
      f"F1 la pragul Exp. 3 diferă în {n_f1_changed} din {len(repro)} cazuri (seed, p)")

## 11. Rezultate: medie ± std pe 5 seeds și diferențe pereche (Exp. 8 − Exp. 3)

Diferențele sunt **pereche**: același seed, același scenariu, aceleași intrări și ponderi. Pentru MAE / RMSE „mai bine” înseamnă mai mic.

In [ ]:
METRICS = ["AUPRC", "AUROC", "Spearman rho", "precision", "recall", "F1", "MCC", "TP", "FP", "FN", "TN"]
PERTURBATION = ["MAE scor", "RMSE scor", "Δ ⟨Z0⟩ medie", "|Δ ⟨Z0⟩| medie"]
LOWER_IS_BETTER = {"MAE scor", "RMSE scor", "|Δ ⟨Z0⟩| medie", "FP", "FN", "timp scorare [s]"}


def mean_std(values, digits=4):
    values = np.asarray(values, dtype=float)
    return f"{values.mean():.{digits}f} ± {values.std(ddof=1):.{digits}f}"


def per_seed(scenario, circuit, p, metric):
    sel = runs[(runs["scenariu"] == scenario) & (runs["circuit"] == circuit) & (runs["p"] == p)]
    return sel.set_index("seed").loc[SEEDS, metric].astype(float)


summary = pd.DataFrame([
    {"scenariu": sc, "circuit": CIRCUIT_LABELS[c], "p": p,
     **{m: mean_std(per_seed(sc, c, p, m), 1 if m in {"TP", "FP", "FN", "TN"} else 4) for m in METRICS},
     "F1 [prag reoptimizat]": mean_std(per_seed(sc, c, p, "F1 [prag reoptimizat]"))}
    for sc in GATE_SETS for c in CIRCUITS for p in P_ALL
]).set_index(["scenariu", "circuit", "p"])
summary.to_csv(RESULTS_DIR / "val_summary.csv")
print("Validation, medie ± std pe 5 seeds (metrici de prag la pragul Exp. 3):")
display(summary[["AUPRC", "AUROC", "Spearman rho", "F1", "MCC", "TP", "FP", "F1 [prag reoptimizat]"]])

impact_rows = []
for sc in GATE_SETS:
    for c in CIRCUITS:
        auprc0 = per_seed(sc, c, 0.0, "AUPRC")
        for p in P_ALL:
            impact_rows.append({"scenariu": sc, "circuit": CIRCUIT_LABELS[c], "p": p,
                                "locuri de zgomot": int(STRUCT.loc[(sc, CIRCUIT_LABELS[c]), "locuri de zgomot"]),
                                **{m: mean_std(per_seed(sc, c, p, m), 6) for m in PERTURBATION},
                                "Spearman rho": mean_std(per_seed(sc, c, p, "Spearman rho")),
                                "AUPRC − AUPRC(p = 0)": mean_std(per_seed(sc, c, p, "AUPRC") - auprc0)})
noise_impact = pd.DataFrame(impact_rows).set_index(["scenariu", "circuit", "p"])
noise_impact.to_csv(RESULTS_DIR / "noise_impact.csv")
print("Perturbarea produsă de zgomot (scoruri cu zgomot vs fără zgomot, același circuit):")
display(noise_impact)

paired_rows, paired_seed_rows = [], []
for sc in GATE_SETS:
    for p in P_ALL:
        for m in ["AUPRC", "AUROC", "Spearman rho", "F1", "MCC", "TP", "FP", *PERTURBATION, "timp scorare [s]"]:
            d = per_seed(sc, "exp8", p, m) - per_seed(sc, "exp3", p, m)
            better = (d < 0) if m in LOWER_IS_BETTER else (d > 0)
            paired_rows.append({"scenariu": sc, "p": p, "metrică": m, "Δ medie": d.mean(), "Δ std": d.std(ddof=1),
                                "Δ (Exp. 8 − Exp. 3)": mean_std(d, 6), "seeds cu Exp. 8 mai bun": f"{int(better.sum())} / {len(d)}",
                                "seeds egale": int((d == 0).sum())})
            for seed in SEEDS:
                paired_seed_rows.append({"scenariu": sc, "p": p, "metrică": m, "seed": seed, "Exp. 3": per_seed(sc, "exp3", p, m)[seed],
                                         "Exp. 8": per_seed(sc, "exp8", p, m)[seed], "Δ": d[seed]})
paired = pd.DataFrame(paired_rows)
paired_seeds = pd.DataFrame(paired_seed_rows)
paired.to_csv(RESULTS_DIR / "paired_differences.csv", index=False)
paired_seeds.to_csv(RESULTS_DIR / "paired_differences_per_seed.csv", index=False)
print("Diferențe pereche la p = 0.01 și 0.05:")
display(paired[paired["p"].isin([0.01, 0.05]) & paired["metrică"].isin(["AUPRC", "Spearman rho", "F1", "MAE scor", "|Δ ⟨Z0⟩| medie"])]
        .set_index(["scenariu", "p", "metrică"])[["Δ (Exp. 8 − Exp. 3)", "seeds cu Exp. 8 mai bun", "seeds egale"]])

## 12. Comparația principală Exp. 3 vs Exp. 8 și cost ledger

In [ ]:
def structure_value(sc, c, key):
    return STRUCT.loc[(sc, CIRCUIT_LABELS[c]), key]


comparison_rows = []
for sc in GATE_SETS:
    for key in ["porți native", "porți 1 qubit", "porți 2 qubiți", "CNOT", "adâncime", "locuri de zgomot"]:
        o, f = structure_value(sc, "exp3", key), structure_value(sc, "exp8", key)
        comparison_rows.append({"scenariu": sc, "metrică": key, "Exp. 3 original": f"{o}", "Exp. 8 fusion": f"{f}",
                                "diferență": f"{f - o:+d} ({(f - o) / o:+.1%})", "seeds cu Exp. 8 mai bun": "structural"})
    for m, p in [("AUPRC", 0.0), ("AUPRC", 0.01), ("AUPRC", 0.05), ("Spearman rho", 0.05), ("MAE scor", 0.05), ("F1", 0.05), ("MCC", 0.05)]:
        o, f = per_seed(sc, "exp3", p, m), per_seed(sc, "exp8", p, m)
        row = paired[(paired["scenariu"] == sc) & (paired["p"] == p) & (paired["metrică"] == m)].iloc[0]
        comparison_rows.append({"scenariu": sc, "metrică": f"{m} la p = {p}" + (" (prag Exp. 3)" if m in {"F1", "MCC"} else ""),
                                "Exp. 3 original": mean_std(o, 6 if m == "MAE scor" else 4), "Exp. 8 fusion": mean_std(f, 6 if m == "MAE scor" else 4),
                                "diferență": row["Δ (Exp. 8 − Exp. 3)"], "seeds cu Exp. 8 mai bun": row["seeds cu Exp. 8 mai bun"]})
    sim = {c: runs[(runs["scenariu"] == sc) & (runs["circuit"] == c) & (runs["p"] > 0)].groupby("seed")["timp scorare [s]"].sum().loc[SEEDS] for c in CIRCUITS}
    comparison_rows.append({"scenariu": sc, "metrică": "timp simulare cu zgomot / seed [s] (3 valori p, 56 962 tranzacții)",
                            "Exp. 3 original": mean_std(sim["exp3"], 1), "Exp. 8 fusion": mean_std(sim["exp8"], 1),
                            "diferență": mean_std(sim["exp8"] - sim["exp3"], 1),
                            "seeds cu Exp. 8 mai bun": f"{int((sim['exp8'] < sim['exp3']).sum())} / {len(SEEDS)}"})
comparison = pd.DataFrame(comparison_rows).set_index(["scenariu", "metrică"])
comparison.to_csv(RESULTS_DIR / "comparison_vs_exp3.csv")
for sc in GATE_SETS:
    print(f"Scenariul {SCENARIO_LABELS[sc]} — validation, medie ± std pe 5 seeds, diferențe pereche")
    display(comparison.loc[sc])

n_val = len(y_val)
ledger_rows = []
for c in CIRCUITS:
    executions = len(P_ALL) * n_val
    ledger_rows.append({
        "circuit": CIRCUIT_LABELS[c],
        "antrenare nouă în Exp. 8 (execuții)": 0,
        "antrenare, dacă s-ar reantrena: 30 × 590 × (1 + 2P) / seed": 30 * 590 * (1 + 2 * 32),
        "execuții / exemplu, Tracker parameter shift": tracker_per_example[c],
        "inferență validation / seed: 4 p × 56 962": executions,
        "inferență validation, 5 seeds": executions * len(SEEDS),
        "shots nominale, 5 seeds (1024 / execuție)": executions * len(SEEDS) * SHOTS_PER_EXECUTION,
        "porți / execuție (A)": structure_value("A", c, "porți native"), "adâncime / execuție (A)": structure_value("A", c, "adâncime"),
        "porți / execuție (B)": structure_value("B", c, "porți native"), "adâncime / execuție (B)": structure_value("B", c, "adâncime"),
        "timp simulare zgomot A, 5 seeds [min]": runs[(runs["scenariu"] == "A") & (runs["circuit"] == c) & (runs["p"] > 0)]["timp scorare [s]"].sum() / 60,
        "timp simulare zgomot B, 5 seeds [min]": runs[(runs["scenariu"] == "B") & (runs["circuit"] == c) & (runs["p"] > 0)]["timp scorare [s]"].sum() / 60,
    })
cost_ledger = pd.DataFrame(ledger_rows).set_index("circuit")
cost_ledger.to_csv(RESULTS_DIR / "cost_ledger.csv")
print("Cost ledger (execuțiile depind de parametri, exemple și valori p, nu de porți; timpul de simulare NU este cost hardware):")
display(cost_ledger.T)

## 13. Grafice

In [ ]:
draw_x, draw_w = torch.tensor(x_probe), models[SEEDS[0]]["weights"]
circuit_text = []
for c, fn in CIRCUITS.items():
    fig, ax = qml.draw_mpl(qml.QNode(fn, dev4), decimals=2, style="pennylane")(draw_x, draw_w)
    fig.suptitle(f"{CIRCUIT_LABELS[c]} · Scenariul A · {int(structure_value('A', c, 'porți native'))} porți, "
                 f"adâncime {int(structure_value('A', c, 'adâncime'))}, {int(structure_value('A', c, 'locuri de zgomot'))} locuri de zgomot "
                 f"(seed 0, prima tranzacție din validation)", fontsize=14)
    fig.savefig(PLOTS_DIR / f"1_circuit_{c}.png", dpi=110, bbox_inches="tight")
    plt.show()
    circuit_text.append(f"=== {CIRCUIT_LABELS[c]} ===\n" + qml.draw(qml.QNode(fn, dev4), decimals=2, max_length=200)(draw_x, draw_w))
(PLOTS_DIR / "1_circuits.txt").write_text("\n\n".join(circuit_text), encoding="utf-8")
print(circuit_text[1])

In [ ]:
def bar_labels(ax, bars, fmt="{:.0f}"):
    for bar in bars:
        if bar.get_height() > 0:
            ax.annotate(fmt.format(bar.get_height()), (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                        textcoords="offset points", xytext=(0, 3), ha="center", fontsize=8, color="0.25")


# 2. porți pe categorii
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, sc in zip(axes, GATE_SETS):
    gates = [g for g in ["Hadamard", "Rot", "RX", "RY", "RZ", "CNOT"]
             if structure_value(sc, "exp3", f"#{g}") + structure_value(sc, "exp8", f"#{g}") > 0]
    xg = np.arange(len(gates))
    for k, c in enumerate(CIRCUITS):
        bars = ax.bar(xg + (k - 0.5) * 0.38, [structure_value(sc, c, f"#{g}") for g in gates], width=0.36,
                      color=CIRCUIT_COLORS[c], label=f"{CIRCUIT_LABELS[c]} ({int(structure_value(sc, c, 'porți native'))} porți)")
        bar_labels(ax, bars)
    ax.set_xticks(xg, gates)
    ax.set_title(f"Scenariul {SCENARIO_LABELS[sc]}")
    ax.legend(loc="upper left", fontsize=9)
axes[0].set_ylabel("număr de porți după decompoziție")
fig.suptitle("Porți pe tipuri: aceleași 21 CNOT; fuziunea schimbă doar porțile de un qubit", fontsize=11)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "2_gate_counts.png", dpi=130)
plt.show()

# 3. locuri de zgomot: canale CNOT (comune) + canale de un qubit
groups = [("A", "A · Rot nativ\n(oficial)"), ("B", "B · Rot → RZ·RY·RZ"), ("B′", "B′ · fără RZ(0)\n(doar structural)")]
fig, ax = plt.subplots(figsize=(8.5, 4.4))
for gi, (sc, label) in enumerate(groups):
    for k, c in enumerate(CIRCUITS):
        xpos_bar = gi + (k - 0.5) * 0.38
        two = 2 * structure_value(sc, c, "porți 2 qubiți")
        one = structure_value(sc, c, "porți 1 qubit")
        ax.bar(xpos_bar, two, width=0.36, color=CNOT_COLOR, label="canale după CNOT (identice)" if gi == 0 and k == 0 else None)
        ax.bar(xpos_bar, one, width=0.36, bottom=two, color=CIRCUIT_COLORS[c], edgecolor="white", linewidth=2,
               label=f"canale după porți de un qubit, {CIRCUIT_LABELS[c]}" if gi == 0 else None)
        ax.annotate(f"{two + one}", (xpos_bar, two + one), textcoords="offset points", xytext=(0, 3), ha="center", fontsize=9, color="0.2")
ax.set_xticks(range(len(groups)), [g[1] for g in groups])
ax.set_ylabel("locuri de zgomot (canale depolarizante)")
ax.set_title("Locuri de zgomot înainte / după fuziune")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "3_noise_locations.png", dpi=130)
plt.show()

In [ ]:
xpos = np.arange(len(P_ALL))
xlabels = ["0", *[str(p) for p in NOISE_LEVELS]]
SERIES = [("A", "exp3", -0.12), ("A", "exp8", -0.04), ("B", "exp3", 0.04), ("B", "exp8", 0.12)]


def plot_vs_p(metric, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(7.5, 4.3))
    for sc, c, dx in SERIES:
        values = [per_seed(sc, c, p, metric) for p in P_ALL]
        ax.errorbar(xpos + dx, [v.mean() for v in values], yerr=[v.std(ddof=1) for v in values], color=CIRCUIT_COLORS[c],
                    linestyle="-" if sc == "A" else "--", marker="o" if c == "exp3" else "s", markersize=8,
                    markerfacecolor=CIRCUIT_COLORS[c] if sc == "A" else "white", linewidth=2, capsize=4,
                    label=f"{CIRCUIT_LABELS[c]} · Scenariul {sc}")
    ax.set_xticks(xpos, xlabels)
    ax.set_xlabel("probabilitate depolarizantă p per poartă")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=130)
    plt.show()


plot_vs_p("AUPRC", "AUPRC (validation)", "AUPRC vs p · medie ± std pe 5 seeds", "4_auprc_vs_p.png")
plot_vs_p("Spearman rho", "Spearman ρ vs scoruri fără zgomot", "Spearman ρ vs p · medie ± std pe 5 seeds", "5_spearman_vs_p.png")
plot_vs_p("MAE scor", "MAE |scor cu zgomot − scor fără zgomot|", "MAE noisy–noiseless vs p · medie ± std pe 5 seeds", "6_mae_vs_p.png")

In [ ]:
# 7. Δ AUPRC per seed la p = 0.05
fig, ax = plt.subplots(figsize=(8, 4.2))
for k, sc in enumerate(GATE_SETS):
    d = per_seed(sc, "exp8", 0.05, "AUPRC") - per_seed(sc, "exp3", 0.05, "AUPRC")
    bars = ax.bar(np.arange(len(SEEDS)) + (k - 0.5) * 0.38, d.to_numpy(), width=0.36,
                  color="#4a4f57" if sc == "A" else "white", edgecolor="#4a4f57", hatch=None if sc == "A" else "///",
                  label=f"Scenariul {SCENARIO_LABELS[sc]} (medie {d.mean():+.4f})")
    ax.axhline(d.mean(), color="#4a4f57", linestyle="-" if sc == "A" else "--", linewidth=1)
ax.axhline(0, color="0.3", linewidth=1)
ax.set_xticks(range(len(SEEDS)), [f"seed {s}" for s in SEEDS])
ax.set_ylabel("Δ AUPRC (Exp. 8 − Exp. 3)")
ax.set_title("Diferența AUPRC per seed la p = 0.05 (> 0: fuziunea ajută) · validation")
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "7_delta_auprc_per_seed_p0.05.png", dpi=130)
plt.show()

# 8. Scenariul A vs Scenariul B
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
xs = np.arange(len(GATE_SETS))
for k, c in enumerate(CIRCUITS):
    bars = axes[0].bar(xs + (k - 0.5) * 0.38, [structure_value(sc, c, "locuri de zgomot") for sc in GATE_SETS], width=0.36,
                       color=CIRCUIT_COLORS[c], label=CIRCUIT_LABELS[c])
    bar_labels(axes[0], bars)
    for sc_i, sc in enumerate(GATE_SETS):
        v = per_seed(sc, c, 0.05, "MAE scor")
        xb = sc_i + (k - 0.5) * 0.38
        axes[1].bar(xb, v.mean(), width=0.36, color=CIRCUIT_COLORS[c], yerr=v.std(ddof=1), capsize=4, label=CIRCUIT_LABELS[c] if sc_i == 0 else None)
        axes[1].scatter(np.full(len(v), xb), v, s=14, color=SEED_DOT_COLOR, zorder=3)
for sc_i, sc in enumerate(GATE_SETS):
    for m_i, (m, marker) in enumerate([("AUPRC", "o"), ("Spearman rho", "s")]):
        d = per_seed(sc, "exp8", 0.05, m) - per_seed(sc, "exp3", 0.05, m)
        xb = sc_i + (m_i - 0.5) * 0.3
        axes[2].errorbar(xb, d.mean(), yerr=d.std(ddof=1), fmt=marker, color="#4a4f57", markersize=8, capsize=4,
                         label=f"Δ {m}" if sc_i == 0 else None)
        axes[2].scatter(np.full(len(d), xb), d, s=14, color=SEED_DOT_COLOR, zorder=3)
axes[2].axhline(0, color="0.3", linestyle="--", linewidth=1)
for ax, title in zip(axes, ["locuri de zgomot", "MAE noisy–noiseless la p = 0.05", "Δ (Exp. 8 − Exp. 3) la p = 0.05, per seed"]):
    ax.set_xticks(xs, [SCENARIO_LABELS[sc] for sc in GATE_SETS])
    ax.set_title(title)
    ax.legend(fontsize=8)
fig.suptitle("Scenariul A (Rot nativ) vs Scenariul B (Rot descompus): avantajul fuziunii depinde de cum este compilat Rot", fontsize=11)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "8_scenario_A_vs_B.png", dpi=130)
plt.show()
print("grafice salvate în", PLOTS_DIR.relative_to(REPO_ROOT))

## 14. Cum citim rezultatele

- **Echivalența** este condiția de validitate: la p = 0 cele două circuite sunt aceeași funcție (secțiunea 6), deci orice diferență sub zgomot vine **doar** din numărul de canale depolarizante.
- **Scenariul A** este comparația principală (modelul oficial de simulare). **Scenariul B** arată ce rămâne din avantaj dacă `Rot` nu este o singură operație fizică. Rezultatele celor două scenarii nu se amestecă.
- Reducerea porților nu este același lucru cu reducerea perturbării: citim separat structura (secțiunea 7), MAE / ρ (perturbarea) și AUPRC (clasificarea).
- Pe validation (99 de fraude), 5 seeds; metricile de prag folosesc pragurile Exp. 3. Testul nu a fost folosit.
- Raportul final, cu interpretarea, este în `results/result_exp8/report.md`.